In [1]:
import pandas as pd

df = pd.read_csv("../data/ev_charging_patterns.csv")

print(df.shape)
df.head()

(1320, 20)


,User ID,Vehicle Model,Battery Capacity (kWh),Charging Station ID,Charging Station Location,Charging Start Time,Charging End Time,Energy Consumed (kWh),Charging Duration (hours),Charging Rate (kW),Charging Cost (USD),Time of Day,Day of Week,State of Charge (Start %),State of Charge (End %),Distance Driven (since last charge) (km),Temperature (°C),Vehicle Age (years),Charger Type,User Type
0,User_1,BMW i3,108.463007,Station_391,Houston,2024-01-01 00:00:00,2024-01-01 00:39:00,60.712346,0.591363,36.389181,13.087717,Evening,Tuesday,29.371576,86.119962,293.602111,27.947953,2.0,DC Fast Charger,Commuter
1,User_2,Hyundai Kona,100.000000,Station_428,San Francisco,2024-01-01 01:00:00,2024-01-01 03:01:00,12.339275,3.133652,30.677735,21.128448,Morning,Monday,10.115778,84.664344,112.112804,14.311026,3.0,Level 1,Casual Driver
2,User_3,Chevy Bolt,75.000000,Station_181,San Francisco,2024-01-01 02:00:00,2024-01-01 04:48:00,19.128876,2.452653,27.513593,35.667270,Morning,Thursday,6.854604,69.917615,71.799253,21.002002,2.0,Level 2,Commuter
3,User_4,Hyundai Kona,50.000000,Station_327,Houston,2024-01-01 03:00:00,2024-01-01 06:42:00,79.457824,1.266431,32.882870,13.036239,Evening,Saturday,83.120003,99.624328,199.577785,38.316313,1.0,Level 1,Long-Distance Traveler
4,User_5,Hyundai Kona,50.000000,Station_108,Los Angeles,2024-01-01 04:00:00,2024-01-01 05:46:00,19.629104,2.019765,10.215712,10.161471,Morning,Saturday,54.258950,63.743786,203.661847,-7.834199,1.0,Level 1,Long-Distance Traveler


In [2]:
print(df.columns.tolist())

['User ID', 'Vehicle Model', 'Battery Capacity (kWh)', 'Charging Station ID', 'Charging Station Location', 'Charging Start Time', 'Charging End Time', 'Energy Consumed (kWh)', 'Charging Duration (hours)', 'Charging Rate (kW)', 'Charging Cost (USD)', 'Time of Day', 'Day of Week', 'State of Charge (Start %)', 'State of Charge (End %)', 'Distance Driven (since last charge) (km)', 'Temperature (°C)', 'Vehicle Age (years)', 'Charger Type', 'User Type']


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1320 entries, 0 to 1319
Data columns (total 20 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   User ID                                   1320 non-null   str    
 1   Vehicle Model                             1320 non-null   str    
 2   Battery Capacity (kWh)                    1320 non-null   float64
 3   Charging Station ID                       1320 non-null   str    
 4   Charging Station Location                 1320 non-null   str    
 5   Charging Start Time                       1320 non-null   str    
 6   Charging End Time                         1320 non-null   str    
 7   Energy Consumed (kWh)                     1254 non-null   float64
 8   Charging Duration (hours)                 1320 non-null   float64
 9   Charging Rate (kW)                        1254 non-null   float64
 10  Charging Cost (USD)                       1320 

In [4]:
charging_df = df.copy()

charging_df.columns = (
    charging_df.columns
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
    .str.replace("%", "percent", regex=False)
    .str.replace("°", "", regex=False)
)

charging_df.columns.tolist()

['user_id',
 'vehicle_model',
 'battery_capacity_kwh',
 'charging_station_id',
 'charging_station_location',
 'charging_start_time',
 'charging_end_time',
 'energy_consumed_kwh',
 'charging_duration_hours',
 'charging_rate_kw',
 'charging_cost_usd',
 'time_of_day',
 'day_of_week',
 'state_of_charge_start_percent',
 'state_of_charge_end_percent',
 'distance_driven_since_last_charge_km',
 'temperature_c',
 'vehicle_age_years',
 'charger_type',
 'user_type']

In [5]:
charging_df["soc_increase"] = (
    charging_df["state_of_charge_end_percent"] -
    charging_df["state_of_charge_start_percent"]
)

In [6]:
charging_df["soc_increase"] = (
    charging_df["state_of_charge_end_percent"] -
    charging_df["state_of_charge_start_percent"]
)

In [7]:
charging_df["is_fast_charge"] = (
    charging_df["charger_type"]
    .str.lower()
    .str.contains("fast")
).astype(int)

In [8]:
charging_df["charging_risk_score"] = 0

charging_df.loc[
    charging_df["is_fast_charge"] == 1,
    "charging_risk_score"
] += 35

charging_df.loc[
    charging_df["state_of_charge_end_percent"] > 90,
    "charging_risk_score"
] += 25

charging_df.loc[
    charging_df["temperature_c"] > 35,
    "charging_risk_score"
] += 25

charging_df.loc[
    charging_df["charging_rate_kw"] > 100,
    "charging_risk_score"
] += 15

In [9]:
def risk_level(score):
    if score >= 70:
        return "High"
    elif score >= 40:
        return "Medium"
    else:
        return "Low"


charging_df["charging_risk_level"] = (
    charging_df["charging_risk_score"]
    .apply(risk_level)
)

In [10]:
charging_df[
    [
        "user_id",
        "vehicle_model",
        "charger_type",
        "charging_rate_kw",
        "state_of_charge_start_percent",
        "state_of_charge_end_percent",
        "soc_increase",
        "temperature_c",
        "charging_risk_score",
        "charging_risk_level"
    ]
].head(10)

,user_id,vehicle_model,charger_type,charging_rate_kw,state_of_charge_start_percent,state_of_charge_end_percent,soc_increase,temperature_c,charging_risk_score,charging_risk_level
0,User_1,BMW i3,DC Fast Charger,36.389181,29.371576,86.119962,56.748386,27.947953,35,Low
1,User_2,Hyundai Kona,Level 1,30.677735,10.115778,84.664344,74.548566,14.311026,0,Low
2,User_3,Chevy Bolt,Level 2,27.513593,6.854604,69.917615,63.063011,21.002002,0,Low
3,User_4,Hyundai Kona,Level 1,32.882870,83.120003,99.624328,16.504325,38.316313,50,Medium
4,User_5,Hyundai Kona,Level 1,10.215712,54.258950,63.743786,9.484836,-7.834199,0,Low
5,User_6,Nissan Leaf,DC Fast Charger,14.334523,75.217748,71.982288,-3.235461,-5.274218,35,Low
6,User_7,Chevy Bolt,Level 2,26.185188,60.751781,70.796097,10.044316,27.551335,0,Low
7,User_8,Chevy Bolt,Level 2,26.702908,56.201703,63.786815,7.585112,-4.417460,0,Low
8,User_9,Chevy Bolt,Level 1,14.294923,33.466200,92.961421,59.495221,22.516706,25,Low
9,User_10,Hyundai Kona,DC Fast Charger,11.761000,27.399455,70.053381,42.653927,27.512019,35,Low


In [11]:
charging_df.to_csv(
    "../models/charging_risk_features.csv",
    index=False
)

In [12]:
charging_df["charging_risk_level"].value_counts()

charging_risk_level
Low       1162
Medium     147
High        11
Name: count, dtype: int64

In [13]:
charging_summary = {
    "avg_duration": charging_df["charging_duration_hours"].mean(),
    "avg_soc_increase": charging_df["soc_increase"].mean(),
    "fast_charge_ratio": charging_df["is_fast_charge"].mean() * 100,
    "avg_risk_score": charging_df["charging_risk_score"].mean()
}

charging_summary

{'avg_duration': np.float64(2.2693773889553506),
 'avg_soc_increase': np.float64(26.01157808613296),
 'fast_charge_ratio': np.float64(32.57575757575758),
 'avg_risk_score': np.float64(19.583333333333332)}

In [14]:
charging_df.groupby(
    "charging_risk_level"
).size()

charging_risk_level
High        11
Low       1162
Medium     147
dtype: int64

In [15]:
charging_summary_df = pd.DataFrame([charging_summary])

charging_summary_df.to_csv(
    "../models/charging_summary.csv",
    index=False
)

In [17]:
risk_counts = (
    charging_df["charging_risk_level"]
    .value_counts()
    .reset_index()
)

risk_counts.columns = [
    "risk_level",
    "count"
]

risk_counts.to_csv(
    "../models/charging_risk_distribution.csv",
    index=False
)